In [1]:
#DESCRIPCIÓN:Bases nuevas mediante imputación de valores faltantes con Random Forest+error OOB

# Pandas se usa para leer archivos Excel y trabajar con tablas de datos
import pandas as pd
# Numpy se usa para cálculos numéricos y para representar valores faltantes
import numpy as np
# RandomForestRegressor es el modelo que se utiliza para predecir los valores ausentes
from sklearn.ensemble import RandomForestRegressor
# Estas clases sirven para indicar mejor qué tipo de datos recibe o devuelve cada función
from typing import Dict, Optional, Tuple


class MissForestContinuous:
    
    #Clase para realizar una imputación tipo MissForest en bases numéricas.
    #La clase va columna por columna. Cuando encuentra una variable con valores
    #faltantes, entrena un Random Forest usando como explicativas el resto de
    #variables y después predice los valores que faltan.
    #Está pensada para variables continuas, por eso se utiliza RandomForestRegressor.

    def __init__(
        self,
        n_estimators: int = 300,
        max_iter: int = 10,
        min_iter: int = 2,
        max_features: str | int | float = 0.8,
        random_state: int = 42,
        n_jobs: int = -1,
        verbose: bool = False,
    ) -> None:
        #parámetros del modelo

        # número de árboles que tendrá cada Random Forest
        self.n_estimators = n_estimators

        # número máximo de vueltas completas del algoritmo
        self.max_iter = max_iter

        # número mínimo de iteraciones antes de permitir que el algoritmo pare
        self.min_iter = min_iter

        # número de variables que puede usar cada árbol en cada división
        self.max_features = max_features

        # semilla para que los resultados sean reproducibles
        self.random_state = random_state

        # Número de núcleos del ordenador que se usarán. Con -1 se usan todos los disponibles
        self.n_jobs = n_jobs

        # si verbose=True, se podrían añadir mensajes para seguir el proceso
        self.verbose = verbose

        # diccionario donde se guardará el error OOB de cada variable imputada
        self.oob_errors_: Dict[str, float] = {}

        # error OOB medio de todas las variables imputadas
        self.oob_error_global_: Optional[float] = None

        # aquí se guarda la evolución del cambio entre iteraciones
        self.delta_history_: list[float] = []

        # número final de iteraciones ejecutadas
        self.iterations_run_: int = 0

        # lista de columnas que han tenido valores faltantes y han sido imputadas
        self.columns_imputed_: list[str] = []


    def _initial_imputation(self, df: pd.DataFrame) -> pd.DataFrame:
        
        #hace una primera imputación sencilla con la media.

        #Esto se necesita porque Random Forest no puede entrenarse si todavía
        #hay valores faltantes en las variables explicativas.
        

        # copiamos la base para no modificar la original
        df_init = df.copy()

        # se recorre cada columna y se rellena sus huecos con la media
        for col in df_init.columns:
            mean_value = df_init[col].mean()
            df_init[col] = df_init[col].fillna(mean_value)

        # se devuelve la base inicial ya sin huecos
        return df_init


    def _compute_delta(
        self,
        new_df: pd.DataFrame,
        old_df: pd.DataFrame,
        cols: list[str]
    ) -> float:
        
        #calcula cuánto cambia la base imputada entre una iteración y otra.

        #si el cambio es muy pequeño, significa que el algoritmo ya está estabilizándose.
       

        # diferencia cuadrática entre la base nueva y la anterior
        numerator = ((new_df[cols] - old_df[cols]) ** 2).to_numpy().sum()

        # escalamos por el tamaño de los valores imputados
        denominator = (new_df[cols] ** 2).to_numpy().sum()

        # evitamos dividir entre cero
        if denominator == 0:
            return 0.0

        # devolvemos la variación relativa
        return float(numerator / denominator)


    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
       
        #Aplica la imputación a una base de datos.

        #Recibe un DataFrame numérico con valores faltantes y devuelve otro
        #DataFrame con los valores imputados.
        

        # Comprobamos que la entrada sea realmente un DataFrame
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df debe ser un pandas.DataFrame")

        # No tiene sentido imputar una base vacía
        if df.empty:
            raise ValueError("El dataframe está vacío")

        # Comprobamos que todas las columnas sean numéricas
        non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]

        # Si hay columnas no numéricas, se avisa para limpiarlas antes
        if non_numeric:
            raise ValueError(
                f"Todas las columnas deben ser numéricas. Columnas no numéricas: {non_numeric}"
            )

        # Guardamos una copia de la base original con sus valores faltantes
        original_df = df.copy()

        # Localizamos las columnas que tienen al menos un valor faltante
        cols_with_na = [
            col for col in original_df.columns
            if original_df[col].isna().sum() > 0
        ]

        # Guardamos esas columnas como información del proceso
        self.columns_imputed_ = cols_with_na.copy()

        # Si no hay valores faltantes, no hace falta imputar nada
        if len(cols_with_na) == 0:
            self.oob_errors_ = {}
            self.oob_error_global_ = np.nan
            self.delta_history_ = []
            self.iterations_run_ = 0
            return original_df.copy()

        # Ordenamos las columnas de menor a mayor número de valores faltantes
        # Esta es una idea típica de MissForest: empezar por las columnas más completas
        sorted_cols = sorted(
            cols_with_na,
            key=lambda c: original_df[c].isna().sum()
        )

        # Hacemos una primera imputación simple con la media
        df_imp = self._initial_imputation(original_df)

        # Guardamos la mejor base encontrada hasta el momento
        best_df = df_imp.copy()

        # Al principio, el mejor cambio es infinito para que la primera iteración mejore
        best_delta = np.inf

        # Aquí se guardarán los mejores errores OOB encontrados
        best_oob_errors: Dict[str, float] = {}

        # Repetimos el proceso varias veces, hasta un máximo de max_iter
        for it in range(self.max_iter):

            # Guardamos la base antes de empezar esta iteración
            df_old = df_imp.copy()

            # Diccionario temporal para los errores OOB de esta iteración
            oob_errors_iter: Dict[str, float] = {}

            # Recorremos las columnas con valores faltantes
            for col in sorted_cols:

                # Marcamos las filas donde esa variable falta
                missing_mask = original_df[col].isna()

                # Marcamos las filas donde esa variable sí está observada
                obs_mask = ~missing_mask

                # Para entrenar, usamos como X todas las variables menos la que queremos imputar
                X_train = df_imp.loc[obs_mask].drop(columns=[col])

                # Como y usamos la columna objetivo en las filas observadas
                y_train = df_imp.loc[obs_mask, col]

                # Para predecir, usamos las filas donde falta la variable objetivo
                X_pred = df_imp.loc[missing_mask].drop(columns=[col])

                # Si hay muy pocos datos observados, no entrenamos el modelo para esa columna
                if len(y_train) < 2:
                    continue

                # Definimos el Random Forest para esta variable
                rf = RandomForestRegressor(
                    n_estimators=self.n_estimators,
                    max_features=self.max_features,
                    bootstrap=True,
                    oob_score=True,
                    random_state=self.random_state,
                    n_jobs=self.n_jobs,
                )

                # Entrenamos el modelo con las filas donde la variable está observada
                rf.fit(X_train, y_train)

                # Si realmente hay valores faltantes, los predecimos
                if X_pred.shape[0] > 0:
                    y_pred = rf.predict(X_pred)
                    df_imp.loc[missing_mask, col] = y_pred

                # Si el modelo tiene predicciones OOB, calculamos el error OOB
                if hasattr(rf, "oob_prediction_") and rf.oob_prediction_ is not None:

                    # Predicciones OOB para las observaciones de entrenamiento
                    oob_pred = rf.oob_prediction_

                    # Varianza de la variable original
                    var_y = np.var(y_train, ddof=1)

                    # Calculamos el NRMSE OOB solo si la varianza no es cero
                    if var_y > 0:
                        nrmse_oob = np.sqrt(
                            np.mean((y_train - oob_pred) ** 2) / var_y
                        )

                        # Guardamos el error para esta columna
                        oob_errors_iter[col] = float(nrmse_oob)

            # Calculamos cuánto ha cambiado la base imputada en esta iteración
            delta = self._compute_delta(df_imp, df_old, sorted_cols)

            # Guardamos la evolución del cambio
            self.delta_history_.append(delta)

            # Si esta iteración mejora la anterior, guardamos esta versión como la mejor
            if delta < best_delta:
                best_delta = delta
                best_df = df_imp.copy()
                best_oob_errors = oob_errors_iter.copy()

            # Parada temprana:
            # si ya hemos hecho el mínimo de iteraciones y el cambio empieza a aumentar,
            # se detiene el proceso para evitar seguir empeorando
            if it + 1 >= self.min_iter and delta > best_delta:
                break

        # Guardamos cuántas iteraciones se han ejecutado finalmente
        self.iterations_run_ = len(self.delta_history_)

        # Guardamos los errores OOB de la mejor iteración
        self.oob_errors_ = best_oob_errors

        # Calculamos el error OOB global como media de los errores por variable
        self.oob_error_global_ = (
            float(np.mean(list(best_oob_errors.values())))
            if best_oob_errors else np.nan
        )

        # Devolvemos la mejor base imputada encontrada
        return best_df


def preparar_dataframe_para_missforest(
    path_excel: str,
    sheet_name: Optional[int | str] = 0,
    date_column: Optional[str] = "Fecha",
    na_values_extra: Optional[list[str]] = None,
) -> Tuple[pd.DataFrame, Optional[pd.Series], pd.DataFrame]:
    ¡
    #Lee un archivo Excel y prepara la base para poder aplicar MissForestContinuous.

    #Devuelve 
    #1. La base original sin la fecha.
    #2. La columna de fecha, si existe.
    #3. La base convertida a formato numérico.


    # Valores que en las bases pueden aparecer como faltantes
    if na_values_extra is None:
        na_values_extra = [
            "N/D", "ND", ".", "********",
            "#N/A", "NA", "na", "n/a", ""
        ]

    # Leemos el Excel
    df = pd.read_excel(path_excel, sheet_name=sheet_name, engine="openpyxl")

    # Si el Excel devuelve varias hojas, nos quedamos con la primera
    if isinstance(df, dict):
        df = list(df.values())[0]

    # Convertimos los valores raros de missing en np.nan
    df = df.replace(na_values_extra, np.nan)

    # Guardamos la fecha aparte si existe
    fecha = None

    if date_column is not None and date_column in df.columns:
        fecha = df[date_column]
        df = df.drop(columns=[date_column])

    # Convertimos todo lo posible a formato numérico
    df_numerico = df.apply(pd.to_numeric, errors="coerce")

    # Devolvemos la base original, la fecha y la base numérica
    return df.copy(), fecha, df_numerico


def guardar_resultado_imputado(
    df_imputado: pd.DataFrame,
    output_path: str,
    fecha: Optional[pd.Series] = None,
    date_column: str = "Fecha",
) -> pd.DataFrame:
    
    #Guarda la base imputada en Excel.

    #Si había columna de fecha, la vuelve a colocar al principio.
    

    # Si existe fecha, la añadimos otra vez como primera columna
    if fecha is not None:
        df_final = pd.concat([fecha.rename(date_column), df_imputado], axis=1)

    # Si no había fecha, se guarda directamente la base imputada
    else:
        df_final = df_imputado.copy()

    # Guardamos el resultado final en Excel
    df_final.to_excel(output_path, index=False, engine="openpyxl")

    # Devolvemos la base final por si se quiere usar después
    return df_final

C:\Users\guill\.anaconda\Nueva carpeta\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\guill\.anaconda\Nueva carpeta\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
#Aqui se va cambiando el archivo de entrada y de salida para cada una de las 18 bases de datos, una vez cada una

archivo_entrada = r"C:\Users\guill\TFG\BASESDEDATOS\Arreglados\AUSTRALIAModBC.xlsx"
archivo_salida  = r"C:\Users\guill\TFG\BASESDEDATOS\Arreglados\AUSTRALIAModBC1.xlsx"

_, fecha, df_num = preparar_dataframe_para_missforest(
    path_excel=archivo_entrada,
    sheet_name=0,
    date_column="Fecha"
)

imputador = MissForestContinuous(
    n_estimators=300,
    max_iter=10,
    min_iter=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=False
)

df_imputado = imputador.fit_transform(df_num)

guardar_resultado_imputado(
    df_imputado=df_imputado,
    output_path=archivo_salida,
    fecha=fecha,
    date_column="Fecha"
)

print(f"OOB global de AUSTRALIAModBC.xlsx: {imputador.oob_error_global_:.6f}")

OOB global de AUSTRALIAModBC.xlsx: 0.460623
